# ODP Socket Probe

这个 notebook 用来一块一块验证 ODP 的 socket 命令链。

建议顺序：
1. 先运行参数单元
2. 再运行辅助函数单元
3. 然后从 `*IDN?` 开始逐块执行
4. 把每块输出结果发回来

In [ ]:
HOST = "192.168.1.1"
PORT = 4196
CHANNEL = "CH1"
TIMEOUT_SECONDS = 3.0
WRITE_TERMINATION = "\n"
READ_TERMINATION = "\n"


In [ ]:
import socket

def send_scpi(command: str, expect_response: bool = True):
    print(f"SEND: {command}")
    chunks = []
    with socket.create_connection((HOST, PORT), timeout=TIMEOUT_SECONDS) as sock:
        sock.settimeout(TIMEOUT_SECONDS)
        sock.sendall((command + WRITE_TERMINATION).encode())

        if not expect_response:
            print("RESULT: sent")
            return None

        terminator = READ_TERMINATION.encode() if READ_TERMINATION else None
        while True:
            try:
                data = sock.recv(4096)
            except socket.timeout:
                break

            if not data:
                break

            chunks.append(data)
            joined = b"".join(chunks)

            if terminator and joined.endswith(terminator):
                break
            if not terminator and len(data) < 4096:
                break

    response = b"".join(chunks).decode(errors="ignore").strip()
    print(f"RECV: {response}")
    return response


## 1. 身份查询

In [ ]:
send_scpi("*IDN?")

## 2. 选择通道

In [ ]:
send_scpi(f"INST {CHANNEL}", expect_response=False)

## 3. 设电压

In [ ]:
send_scpi("VOLT 5", expect_response=False)

## 4. 设电流

In [ ]:
send_scpi("CURR 0.5", expect_response=False)

## 5. 打开输出

In [ ]:
send_scpi("OUTP ON", expect_response=False)

## 6. 读取测量值

In [ ]:
send_scpi("MEAS:VOLT?")

In [ ]:
send_scpi("MEAS:CURR?")

## 7. 关闭输出

In [ ]:
send_scpi("OUTP OFF", expect_response=False)